# Qt–Ft agglomeration: control panel

1. Set **all** parameters in the cell below (LAMMPS `units nano`: nm, ns, zJ, pN, ag).
2. Run it: the derived values (γ, stability α, binding probability, …) and checks are printed.
3. Build the run folder, start the simulation in the background, follow its progress.

The run folder (`runs/single/<run name>/` for one replica, `runs/ensemble/<run name>/` for
several, inside the repository but not in git) holds `params.json` and one `rep_XX/in.lammps` per replica;
it is the record of what was run, not this notebook.

In [ ]:
%matplotlib inline
%load_ext autoreload
# re-import changed qtft code before every cell
%autoreload 2
from qtft import Params, build_run, start, stop, progress, watch, fetch, quicklook

## 1. Parameters

In [ ]:
p = Params(
    # --- system
    qt_per_um3=20.0,           # Qt encapsulins per um^3 of box (number = density x volume, rounded)
    ft_per_um3=80.0,           # Ft ferritins per um^3 of box (4x Qt)
    box_length=4000.0,         # box [nm]: one edge (cube) or [Lx, Ly, Lz]; 4 um cube = 64 um^3
    boundary="periodic",       # "periodic", "reflect" (hard walls) or "harmonic" (soft walls) on all faces
    wall_k_qt=15.0,            # harmonic walls only: E = k (r - d)^2 [zJ/nm^2]; alpha ~ 0.7 at dt 500 ns
    wall_k_ft=6.0,

    # --- particles
    radius_qt=27.5,            # effective radius [nm]
    radius_ft=13.0,            # effective radius [nm]
    D_qt=2e-4,                 # diffusion coefficient [nm^2/ns]  -> gamma = kT/D
    D_ft=5e-4,                 # diffusion coefficient [nm^2/ns]
    temperature=310.0,         # [K]

    # --- soft repulsion E = k (r_c - r)^2, r_c = r_i + r_j   [zJ/nm^2]; all alpha ~ 0.7 at dt 500 ns
    k_qq=7.5,
    k_ff=3.0,
    k_qf=4.3,

    # --- harmonic bond E = K (r - r0)^2
    bond_k=4.3,                # [zJ/nm^2]
    bond_r0=None,              # [nm]; None -> r_Qt + r_Ft

    # --- binding (fix bond/create)
    k_on=1.86e-6,              # binding rate per Qt-Ft pair in contact at full calmodulin loading (f = 1) [1/ns]
    bind_radius=None,          # [nm]; None -> r_Qt + r_Ft
    bind_every=1,              # attempt every N steps
    maxbond_qt=40,             # max bonds per Qt
    maxbond_ft=10,             # max bonds per Ft

    # --- calcium control: baseline -> influx -> exponential pump-out (a phase of 0 s is skipped)
    t_baseline_s=10.0,         # resting calcium [s]
    t_influx_s=30.0,           # peak calcium [s]
    t_pumpout_s=150.0,         # calcium pumped out, decays with ca_tau_s [s]
    ca_rest_uM=0.1,            # resting calcium [uM]
    ca_peak_uM=1.0,            # calcium plateau during the influx (light on) [uM]
    ca_rise_s=5.0,             # 99 % of the plateau reached after [s]; 0 = instantaneous step
    ca_tau_s=20.0,             # pump-out time constant [s]
    ca_K_uM=0.5,               # calmodulin loading f = Ca^n / (K^n + Ca^n): half loaded at K [uM]
    ca_hill=3.0,               # Hill coefficient n
    k_off_ca_per_s=8e-3,       # unbinding rate per bond with Ca-loaded calmodulin [1/s] (literature)
    k_off_free_per_s=0.5,      # unbinding rate per bond with Ca-free calmodulin [1/s] (literature; bond_break="random" needs no 0.098 calibration)
    rate_update_every=100_000, # k_on updated every N steps (100_000 x 500 ns = 50 ms)
    break_every=10,           # unbinding attempts every N steps
    bond_break="random",      # "random": every bond breaks independently (fix bond/break/random);
                              # "longest": built-in fix bond/break (only the longest bond, as before)

    # --- run
    timestep=500.0,            # [ns]
    equil_steps=5000,          # equilibration, binding off

    # --- output / in-run analysis
    output_every=2000,         # thermo + in-run analysis in log.lammps [steps] (1 ms)
    traj_every=200_000,        # traj.dump, bonds.dump, rdf, histograms [steps] (100 ms)
    large_cluster_min=5,
    rdf_bins=200,
    rdf_cutoff=None,           # [nm]; None -> 3 (r_Qt + r_Ft)

    # --- technical
    neighbor_skin=20.0,        # [nm]; benchmark: 20 is 20-45 % faster than 10
    extra_special=1000,
    comm_cutoff=None,          # [nm]; None -> automatic
    insert_maxtry=1000,

    # --- execution
    n_replicas=1,
    mpi_procs=8,               # MPI processes per replica (benchmark: 8 for a single replica, 4 for ensembles)
    max_parallel=None,         # replicas at once; None -> local: cores // mpi_procs, LRZ: all replicas
    master_seed=12345,
    data_dir="runs",           # relative -> inside the repository; runs go to single/ or ensemble/
    run_name=None,             # None -> generated from the parameters
    lmp_command="lmp",
    mpirun_command="mpirun --bind-to none",  # no core pinning
    executor="lrz",            # "local": this PC; "lrz": Slurm job on LRZ (docs/install.md, part B)
    slurm_time="6-00:00:00",   # lrz: time limit per replica (D-HH:MM:SS, at most 7 days)
    slurm_mem="4G",            # lrz: memory per replica
)

print("run folder:", p.run_dir)
print(p.summary())

## 2. Build the run folder

In [ ]:
run_dir = build_run(p)          # build_run(p, overwrite=True) replaces an existing folder
print(run_dir)

In [ ]:
# optional: look at the generated LAMMPS input of replica 1
print((run_dir / "rep_01" / "in.lammps").read_text())

## 3. Run (in the background — closing the notebook does not stop it)

`watch(run_dir)` shows the progress live (refreshed every 10 s) until the run has finished; the stop
button only ends the view, not the run. The same information goes to `launcher.log` every minute:
follow it in an Ubuntu terminal with `tail -f <run_dir>/launcher.log`.

**On LRZ** (`executor="lrz"` in section 1): log in once in a terminal with `ssh lrz` (password + 2FA; the
connection stays open for 8 h). `start` then copies the run folder to LRZ and submits it as a Slurm job
array (email at begin, end and failure), `watch` fetches the progress from there (every 30 s), `stop`
cancels the job, and `fetch` downloads the results when it is done. Setup: `docs/install.md`, part B.

In [ ]:
start(run_dir)
watch(run_dir)    # live view; progress(run_dir) gives a one-off table

In [ ]:
#stop(run_dir)                 # uncomment to stop all replicas of this run

In [ ]:
fetch(run_dir)                # LRZ runs: download all results (dumps, logs) when done

## 4. Parameter sweep

One run per combination of the listed values (grid), built from the parameters `p` of section 1. Every run
gets `p.n_replicas` replicas, and all variants use the same seeds (replica i of every variant sees the same
random numbers, so differences come from the parameters, not from noise). All replicas of the sweep run
through one queue, at most `max_parallel` at once (`mpi_procs` each). Execution settings (`n_replicas`,
`mpi_procs`, `max_parallel`, `master_seed`, …) cannot be swept. Derived values are not adapted: changing
`temperature` or `timestep` also changes α.
Runs: `runs/sweeps/<name>/<label>/`; evaluate them in `analysis.ipynb`, section 6.

In [ ]:
from qtft import build_sweep

variants = {                       # parameter -> list of values; every combination is one run
    "temperature": [300.0, 310.0],
    "k_on": [1e-6, 2e-6],
}
sweep = build_sweep(p, variants, name="T_kon")
start(sweep)
watch(sweep)        # live view; stop(sweep) stops everything, stop(sweep / "<label>") one run

To continue with an existing sweep after restarting the kernel:

```python
from qtft import data_path
sweep = data_path() / "sweeps" / "<name>"
```

## 5. Quick look

In [ ]:
# overlap from every 10th frame of traj.dump (≈1 min for 1000 frames of 14 000 particles);
# quicklook(run_dir, overlap=False) shows only the thermo panels (fast)
quicklook(run_dir);

To continue with an existing run after restarting the kernel:

```python
from qtft import data_path
run_dir = data_path() / "ensemble" / "<run name>"   # or "single"
```

## 6. Performance measurement (once per machine)

Runs short productions of the system defined in `p` with 1, 2, 4, … MPI processes per replica (up to the
physical cores), each time filling the machine with replicas. This is done twice: filling all hardware
threads and filling only the physical cores, to see whether hyperthreading helps. Takes a few minutes for
large systems. Use the recommendation to set `mpi_procs` and `max_parallel` above.
Results: `runs/benchmark/<cores>cores/results.csv`.

In [ ]:
import os
import pandas as pd
from qtft.benchmark import benchmark, physical_cores, recommend

bench = pd.concat([
    benchmark(
        p,                                   # system and potentials as set above
        cores=cores,                         # hardware threads to fill with replicas
        steps=2000,                          # production steps per measurement
        variants={"skin 10": {"neighbor_skin": 10.0}, "skin 20": {"neighbor_skin": 20.0}},
    )
    for cores in sorted({os.cpu_count(), physical_cores()}, reverse=True)
], ignore_index=True)
bench.round(2)

In [ ]:
recommend(bench, n_replicas=p.n_replicas)

After a long run: how many 1-2/1-3/1-4 neighbours did atoms really need? `extra_special`
(default 1000) must stay above the maximum.

In [ ]:
from qtft.benchmark import special_need
special_need(run_dir / "rep_01")

## 7. Force-constant sweep (overlap vs. stiffness)

Every pair family sits in a harmonic well; thermal noise pushes particles into each other by a mean depth
√(2/π) · √(k_BT / 2k / (1 − α/2)). Stiffer constants reduce the overlap, but α (∝ k·Δt) grows and widens it
again: at a fixed time step the overlap is smallest at α ≈ 1. This sweep sets all four constants
(`k_qq`, `k_ff`, `k_qf`, `bond_k`) to the same α, runs a small system (400 Qt and 1000 Ft in 0.8 µm³, 25×
the production density) and compares measured and predicted overlap and the binding kinetics.
Runs: `runs/tuning/sweep/alpha<α>_dt<Δt>ns/`; results: `runs/tuning/sweep/sweep_summary.csv`.

In [ ]:
from qtft.tuning import alpha_sweep, sweep_table

base = Params(
    qt_per_um3=500.0, ft_per_um3=1250.0, box_length=928.3,     # 400 Qt and 1000 Ft in 0.8 um^3
    equil_steps=5000, t_baseline_s=0.0, t_influx_s=0.05, t_pumpout_s=0.0,   # 50 ms influx: constant rates
    ca_rise_s=0.0,                             # calcium step: binding at full rate from the start
    output_every=1000, traj_every=1000, neighbor_skin=20.0,
    n_replicas=3, mpi_procs=1, max_parallel=3,
)
# time step -> target alphas; equilibration and segment length are scaled with the time step (same 50 ms)
sweep = alpha_sweep(base, {500.0: [0.3, 0.5, 0.7, 0.9, 1.2], 250.0: [0.9]}, name="sweep")
start(sweep)                                   # one queue for all runs, 3 replicas at once

To evaluate a sweep that was built earlier (e.g. after restarting the kernel):

```python
from qtft import data_path
sweep = data_path() / "tuning" / "sweep"
```

In [ ]:
watch(sweep)

In [ ]:
import matplotlib.pyplot as plt

table = sweep_table(sweep)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
for ax, (family, df) in zip(axes, table.groupby("family", sort=False)):
    dt500 = df[df["timestep [ns]"] == 500.0]
    ax.plot(dt500["alpha"], dt500["predicted depth [nm]"], "k--", label="predicted (Δt 500 ns)")
    ax.plot(dt500["alpha"], dt500["mean depth [nm]"], "o-", label="measured (Δt 500 ns)")
    other = df[df["timestep [ns]"] != 500.0]
    ax.plot(other["alpha"], other["mean depth [nm]"], "s", label="measured (Δt 250 ns)")
    ax.set_title(family.replace("-", "–"))
    ax.set_xlabel("α")
axes[0].set_ylabel("mean overlap depth (nm)")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
table.round(3)